In [18]:
# Password-protected file access (to be used across Jupyter notebooks)

import msoffcrypto # To access password-protected files
import pandas as pd # To read and manipulate data
from io import BytesIO # Temp handling of decrypted files
from getpass import getpass # Get password input w/out encoding

password = getpass("Enter anon file password: ") # Prompt for password
decrypted_file = BytesIO() # Creates temp file in memory

with open(r"C:\Users\karin\Documents\2. Data\Anonymous_Data.xlsx", "rb") as f: # Open the password-protected file
    office_file = msoffcrypto.OfficeFile(f) # Create Officefile object
    office_file.load_key(password=password) # Load password
    office_file.decrypt(decrypted_file) # Decrypt file into memory

df = pd.read_excel(decrypted_file) # Read decrypted file into a DataFrame (DF)
df.shape # Display the shape of the DF (rows and columns)

(2839, 88)

In [19]:
# Rebuild df_binary for attendance feature below

fail_categories = ['Fail Resit', 'Fail Withdraw', 'Repeat without Attendance', 'Repeat with Attendance', 'Complete Repeat'] # Defining categories which count as a fail
df_binary = df[df['Progression Decision'] != 'Trail Progress'].copy() # Create a new DF excluding 'Trail Progress' rows as it is its own edge case
df_binary['initially_failed'] = df_binary['Progression Decision'].isin(fail_categories) # Create new column in new DF: True if student failed and false if they passed

df_binary.shape # Display the shape of the DF (rows and columns)

(2837, 89)

In [20]:
# Attendance Feature: already numeric but accouting for off-site students with null values

df_binary['is_offsite'] = df_binary['Attendance (%)'].isnull() # Create new column which turns true if attendance is null (off-site student)
df_binary['is_offsite'].value_counts() # Count the number of off-site students (True) and on-site students (False)

is_offsite
False    2785
True       52
Name: count, dtype: int64

Checked the data and realised there was a slight error with column alignment and matching student ID's. Corrected it which enabled further distinction of this number to split by administrative student status details.

In [21]:
# Attendance Feature: check if any off-site students are categorised as on-site via admin student status

df_binary['no_attendance_data'] = df_binary['Attendance (%)'].isnull() # Create new column which turns true if attendance is null (off-site student)
df_binary[df_binary['no_attendance_data']]['Student Status'].value_counts() # Count no. students off-site but categorised via admin student status

Student Status
Off-site     41
PR/Repeat    10
Normal        1
Name: count, dtype: int64

Off-site we have 41 students (expected as 41 were off-site therefore did not have their attendance data recorded), then 10 who were repeat students, presumably with no attendance requirement hence no data. The one normal is an unexplained case.

In [22]:
# Breakdown of student numbers via three student statuses

df_binary['is_offsite'] = df_binary['Student Status'].str.contains('Off-site', case=False, na=False) # Create new column which turns true if student status contains 'Off-site' (off-site student)
df_binary['is_repeating'] = df_binary['Student Status'].str.contains('PR/Repeat', case=False, na=False) # Create new column which turns true if student status contains 'PR/Repeat' (repeating student)
df_binary['unexplained_null_attendance'] = df_binary['no_attendance_data'] & ~df_binary['is_offsite'] & ~df_binary['is_repeating'] # Create new column which turns true if student has no attendance data but is not off-site or repeating

df_binary[['is_offsite', 'is_repeating', 'unexplained_null_attendance']].sum() # Count students in each category

is_offsite                     42
is_repeating                   92
unexplained_null_attendance     1
dtype: int64

Administrative student status does not record this detail and manual adjustments were made during excel data cleaning to clarify the data. The additional off-site students was caught accidentally as they fell outside of the expected cohort of off-site students. Drastic increase in repeating students due to the split between those expected to repeat with attendance or to repeat without having to attend.

In [23]:
# Dividing repeating students into two categories: those with and without attendance expectation

df_binary['is_repeating'] = df_binary['is_repeating'] # All repeating students for general repeat flag
df_binary['repeating_with_no_attendance_expectation'] = df_binary['is_repeating'] & df_binary['Attendance (%)'].isnull() # Those repeating without attendance expectation
df_binary['repeating_with_attendance'] = df_binary['is_repeating'] & df_binary['Attendance (%)'].notnull() # Those repeating with attendance expectation

df_binary[['repeating_with_no_attendance_expectation', 'repeating_with_attendance']].sum() # Count students in each category

repeating_with_no_attendance_expectation    10
repeating_with_attendance                   82
dtype: int64

The above shows the divide between those repeating with expectations of attending and those with no expectation to attend - this is not something that administrative fields pick up and can only be identified due to the combination of multiple datasets. Where a student has null for attendance, the attendance system is not expecting it to pick anything up where as those who should be attending have a record (even if this is 0%).

In [24]:
# Rebuild list of VLE columns

vle_columns = [col for col in df.columns if 'VLE' in col] # List VLE columns
len(vle_columns) # Display the number of VLE columns

19

In [25]:
# VLE Engagement Score

rag_map = {'RED': 0, 'AMBER': 1, 'GREEN': 2} # Ordinal mapping for RAG values, the higher, the better the engagement score

vle_numeric = df_binary[vle_columns].apply(lambda col: col.str.upper()) # Convert all RAG values to uppercase to avoid errors
vle_numeric = vle_numeric.replace(rag_map).replace('GREY', pd.NA) # Replace RAG values with numeric values and replace GREY with null
vle_numeric = vle_numeric.apply(pd.to_numeric, errors='coerce') # Convert all values to numeric

df_binary['vle_avg_score'] = vle_numeric.mean(axis=1) # Calculate the average VLE score for each student across all weeksand store in a new column
df_binary['vle_red_weeks'] = (vle_numeric == 0).sum(axis=1) # Count the number of RED weeks for each student
df_binary['vle_grey_weeks'] = vle_numeric.isnull().sum(axis=1) # Count the number of GREY weeks for each student
df_binary['has_grey_vle'] = df_binary['vle_grey_weeks'] > 3 # Binary flag check if student has more than 3 GREY weeks (adjusted as fewer likely reflects late registration)

df_binary[['vle_avg_score', 'vle_red_weeks', 'vle_grey_weeks', 'has_grey_vle']].describe() # Print the scores

,vle_avg_score,vle_red_weeks,vle_grey_weeks
count,2746.000000,2837.000000,2837.000000
mean,1.315146,3.348255,1.381036
std,0.499003,4.320137,4.126316
min,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000
50%,1.421053,2.000000,0.000000
75%,1.736842,5.000000,0.000000
max,2.000000,19.000000,19.000000


To summarise the above, the VLE average score is 1.32 (from a 0-2 scale) and a median of 1.42 meaning most of the students sit toward the Amber & Green zones so the engagement is quite healthy overall. 

For the red weeks, there is an average of 3.34 weeks per student from the total 19 weeks assessed, however the 75th precentile is only 5 meaning most of the students have only a few but some do have more - this is an expected skew. 

The grey weeks, there is a average of 1.38 and an important note is that the median (50%) is 0.0 meaning that over half of the students have no grey weeks.

In [26]:
# Flag for no VLE data students

df_binary['no_vle_data'] = df_binary['vle_avg_score'].isnull() # Pulls 91 students who's overall VLE engagement is null (grey or non-existant)
df_binary['no_vle_data'].sum() # Check for 91 returned

91

Identified that from the counts listed in 'VLE Engagement Score' above 91 students had null (or GREY) across all 19 VLE weeks. Investigation showed 32 were off-site for the year and 59 were repeating the academic year meaning they were still attached to last year's VLE (the 2023/24 academic year VLE space). Flagged for now and decision on exclusion or seperate group treatment left for modelling stage.

In [27]:
# Departments as categorical confirmation

df_binary['Department'].value_counts() # Confirm there are only 3 departments

Department
Department B    1174
Department A    1042
Department C     621
Name: count, dtype: int64

Confirmed that departments already seen as categorical.

In [28]:
# Admin Status categorical confirmation

df_binary['Admin Status'].value_counts() # Check for clean categories (incl. any typos)

Admin Status
Current Student          2143
Complete & Awarded        535
Withdrawn                 100
Early Exit with Award      44
Dormant Student            10
Suspended Student           4
Transferred Course          1
Name: count, dtype: int64

In [29]:
# Student Status categorical confirmation

df_binary['Student Status'].value_counts() # Check for clean categories (incl. any typos)

Student Status
Normal       2703
PR/Repeat      92
Off-site       42
Name: count, dtype: int64

In [30]:
# Encoding categoricals

categorical_cols = ['Department', 'Student Status', 'Year Group'] # Columns that need to be one-hot encoded - Added Year Group
df_encoded = pd.get_dummies(df_binary, columns=categorical_cols, drop_first=False) # New column created per category as True or False
df_encoded = df_encoded.drop(columns=['Admin Status']) # Drop Admin Status due to data leakage
df_encoded.shape # Check new column count

(2837, 108)

In [31]:
# Run check on columns to confirm

[col for col in df_encoded.columns if 'Admin Status' in col or 'Student Status' in col or 'Department' in col]

['Department_Department A',
 'Department_Department B',
 'Department_Department C',
 'Student Status_Normal',
 'Student Status_Off-site',
 'Student Status_PR/Repeat']

As models need numbers rather than text one-hot encoding was used to turn each category into its own 0/1 column so the model can in turn weigh each category independently without assuming any false orders between them. New counts are as follows:
Row count: 2837 - correct as 2 trail progression not included
Column Count: 110 - additional check ran to ensure all correct columns added

In [32]:
# Final check on rows and columns

df_encoded.shape # Should remain as 2837, 110 (no rows lost or duplicated during the encoding)

(2837, 108)

In [33]:
# Dropping columns used for manual checks

df_encoded = df_encoded.drop(columns=['Notes', 'check'])
df_encoded.shape # check the two columns are dropped

(2837, 106)

In [34]:
# Final check on nulls (or greys)

null_check = df_encoded.isnull().sum() # Count any nulls per column
null_check[null_check > 0] # Show only the columns that have nulls

Attendance (%)                       52
Week 1 & 2 Attendance %             129
Week 2 & 3 Attendance %             106
Week 3 & 4 Attendance %              90
Week 4 & 5 Attendance %              86
Week 5 & 6 Attendance %              86
Week 6 & 7 Attendance %              84
Week 7 & 8 Attendance %              82
Week 8 & 9 Attendance %              81
Week 9 & 10 Attendance %             81
Week 11 & 12 Attendance %           110
Week 11 & 12 Attendance RAG          21
Week 11 & 12 VLE Engagement RAG      21
Week 11 & 12 COMBINED RAG            21
Week 12 & 13 Attendance %           110
Week 12 & 13 Attendance RAG          21
Week 12 & 13 VLE Engagement RAG      21
Week 12 & 13 COMBINED RAG            21
Week 13 & 14 Attendance %           110
Week 13 & 14 Attendance RAG          21
Week 13 & 14 VLE Engagement RAG      21
Week 13 & 14 COMBINED RAG            21
Week 14 & 15 Attendance %           110
Week 14 & 15 Attendance RAG          21
Week 14 & 15 VLE Engagement RAG      21


All final checks passed. Only thing to note is 'Week 11 & 12 VLE Engagement RAG' to 'Week 20 & 21 VLE Engagement RAG' has 21 null categories (no data recorded) - all of these were due to a student Admin Status change (repeating students getting outcomes and normal students taking an early exit award)